# Import Libraries

In [ ]:
import json
import requests
import geopandas as gpd
from shapely.geometry import LineString
from shapely.geometry import shape
import folium
from folium import FeatureGroup
import branca.colormap as cm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import osmnx as ox
from sklearn.cluster import DBSCAN
import numpy as np
import time
import datetime
import pandas as pd
from shapely.geometry import box
from datetime import datetime
import os
from google.colab import drive
import matplotlib.pyplot as plt
import pytz
from libpysal.weights import DistanceBand
from libpysal.weights import KNN
from esda import Moran
from esda import Moran_Local
import libpysal
from splot.esda import moran_scatterplot
import matplotlib.pyplot as plt
from pathlib import Path
import zipfile


# Define Traffic Flow Function for HERE API

In [ ]:
def get_here_traffic_flow_v7(api_key, bbox_str):
    """
    Query HERE Traffic API v7 for flow data within a bounding box.

    Parameters:
        api_key (str): HERE API key
        bbox_str (str): 'west,south,east,north'

    Returns:
        GeoDataFrame: traffic flows with geometry and congestion attributes
    """
    url = "https://data.traffic.hereapi.com/v7/flow"
    params = {
        "in": f"bbox:{bbox_str}",
        "locationReferencing": "shape",
        "apiKey": api_key
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        results = data.get("results", [])
        if not results:
            print(f"⚠️ No results returned for bbox {bbox_str}")
            return None

        records = []
        geometries = []

        for result in results:
            shape_links = result.get("location", {}).get("shape", {}).get("links", [])
            if not shape_links:
                continue

            coords = []
            for link in shape_links:
                points = link.get("points", [])
                coords.extend([(pt["lng"], pt["lat"]) for pt in points])

            if not coords:
                continue

            geometry = LineString(coords)

            flow = result.get("currentFlow", {})
            record = {
                "currentSpeed": flow.get("speed"),
                "freeFlowSpeed": flow.get("freeFlow"),
                "jamFactor": flow.get("jamFactor"),
                "confidence": flow.get("confidence"),
                "roadName": result.get("location", {}).get("description")
            }

            records.append(record)
            geometries.append(geometry)

        if not records:
            print(f"No valid flow segments with shape geometry for bbox {bbox_str}")
            return None

        gdf = gpd.GeoDataFrame(records, geometry=geometries, crs="EPSG:4326")
        print(f"Retrieved {len(gdf)} segments from {bbox_str}")
        return gdf

    except requests.exceptions.RequestException as e:
        print(f"Request failed for bbox {bbox_str}: {e}")
        return None


# Define HERE API key

In [ ]:
HERE_API_KEY = os.getenv("HERE_API_KEY")

if not HERE_API_KEY:
    raise ValueError(
        "HERE_API_KEY environment variable is not configured."
    )

# Pull HERE data for major urban regions in CA

In [ ]:
urban_areas = {
    "San_Francisco": "-122.52,37.70,-122.35,37.84",
    "Los_Angeles": "-118.67,33.70,-118.16,34.34",
    "Sacramento": "-121.60,38.45,-121.30,38.70",
    "San_Diego": "-117.25,32.53,-116.90,33.10",
    "San_Jose": "-122.05,37.20,-121.75,37.45",
    "Fresno": "-119.90,36.65,-119.65,36.85"
}

all_gdfs = []

for city, bbox_str in urban_areas.items():
    print(f"Querying HERE traffic for {city}...")
    gdf = get_here_traffic_flow_v7(HERE_API_KEY, bbox_str)

    if gdf is not None and not gdf.empty:
        gdf["region"] = city
        all_gdfs.append(gdf)

    time.sleep(1.5)  # Sleep to avoid API rate limit

# Combine all collected traffic into one GeoDataFrame
urban_traffic_gdf = gpd.GeoDataFrame(pd.concat(all_gdfs, ignore_index=True), crs="EPSG:4326")
print(f"Total segments collected: {len(urban_traffic_gdf)}")

# Define OSM enrichment function

In [ ]:
def enrich_with_osm_classifications(here_gdf, bbox_tuple):
    """
    Enrich HERE traffic segments with OSM road classifications.

    Parameters:
        here_gdf (GeoDataFrame): Traffic segments from HERE API (EPSG:4326)
        bbox_tuple (tuple): (north, south, east, west)

    Returns:
        GeoDataFrame: HERE data enriched with nearest OSM 'highway' and 'name' attributes.
    """
    # Unpack the bounding box
    north, south, east, west = bbox_tuple

    # Download drivable road network from OSM within the bounding box
    bbox = (west, south, east, north)
    G = ox.graph.graph_from_bbox(bbox, network_type="drive")
    osm_edges = ox.graph_to_gdfs(G, nodes=False)[["name", "highway", "geometry"]]

    # Reproject both HERE and OSM segments to Web Mercator for spatial join
    here_proj = here_gdf.to_crs(epsg=3857)
    osm_proj = osm_edges.to_crs(epsg=3857)

    # Perform nearest spatial join to append OSM attributes to HERE segments
    joined = gpd.sjoin_nearest(here_proj, osm_proj, how="left", distance_col="match_distance")

    # Reproject enriched GeoDataFrame back to WGS84
    enriched = joined.to_crs(epsg=4326)

    return enriched


# Enrich urban areas with OSM highway classifications

In [ ]:
enriched_segments = []

def bbox_str_to_tuple(bbox_str):
    west, south, east, north = map(float, bbox_str.split(","))
    return (north, south, east, west)

for city, bbox_str in urban_areas.items():
    print(f"Enriching OSM data for {city}...")

    region_gdf = urban_traffic_gdf[urban_traffic_gdf["region"] == city].copy()
    bbox_tuple = bbox_str_to_tuple(bbox_str)

    # Clip segments to bounding box
    west, south, east, north = map(float, bbox_str.split(","))
    bbox_geom = box(west, south, east, north)
    region_gdf = region_gdf[region_gdf.geometry.intersects(bbox_geom)]

    enriched = enrich_with_osm_classifications(region_gdf, bbox_tuple)
    enriched["region"] = city
    enriched_segments.append(enriched)

# Merge all enriched tiles into one GeoDataFrame
enriched_urban_traffic_gdf = gpd.GeoDataFrame(
    pd.concat(enriched_segments, ignore_index=True),
    crs="EPSG:4326"
)

print(f"Enriched segments: {len(enriched_urban_traffic_gdf)}")


# Visualize HERE Data with folium map

In [ ]:
def plot_enriched_segments_by_jamfactor(gdf, zoom_start=7, city=None, max_segments=1000):
    """
    Plot HERE+OSM segments by jamFactor color scale using Folium.

    Parameters:
        gdf (GeoDataFrame): Enriched dataset with jamFactor, geometry, etc.
        zoom_start (int): Initial zoom level for Folium map.
        city (str or None): Optional filter to a specific city/region.
        max_segments (int): Max number of segments to plot (for performance).
    """
    if gdf.empty or "jamFactor" not in gdf.columns:
        print("GeoDataFrame is empty or missing jamFactor.")
        return None

    # Filter to one region
    if city:
        gdf = gdf[gdf["region"] == city]
        if gdf.empty:
            print(f"No data available for region: {city}")
            return None

    # Limit number of segments for performance
    gdf = gdf.sample(n=min(len(gdf), max_segments), random_state=42)

    # Center map on geometric center
    center = gdf.unary_union.centroid
    m = folium.Map(location=[center.y, center.x], zoom_start=zoom_start)

    # Define color scale
    colormap = cm.LinearColormap(["green", "yellow", "orange", "red"], vmin=0, vmax=10)
    colormap.caption = "Jam Factor (0 = Free Flow, 10 = Gridlock)"

    for _, row in gdf.iterrows():
        jf = row.get("jamFactor")
        if jf is None or row.geometry is None:
            continue

        coords = [(lat, lon) for lon, lat in row.geometry.coords]
        color = colormap(jf)

        popup_html = (
            f"<b>Region:</b> {row.get('region', 'N/A')}<br>"
            f"<b>Road:</b> {row.get('roadName', 'Unknown')}<br>"
            f"<b>Speed:</b> {row.get('currentSpeed', 'n/a')} km/h<br>"
            f"<b>Free Flow:</b> {row.get('freeFlowSpeed', 'n/a')} km/h<br>"
            f"<b>Jam Factor:</b> {jf}<br>"
            f"<b>Highway Type:</b> {row.get('highway', 'Unknown')}"
        )

        folium.PolyLine(
            locations=coords,
            color=color,
            weight=4,
            popup=folium.Popup(popup_html, max_width=300),
            opacity=0.8
        ).add_to(m)

    colormap.add_to(m)
    return m

# Show 1000 segments from all of California
plot_enriched_segments_by_jamfactor(enriched_urban_traffic_gdf)

# Show 500 segments from Los Angeles only
#plot_enriched_segments_by_jamfactor(enriched_urban_traffic_gdf, city="Los_Angeles", max_segments=500, zoom_start=10)

# Apply K-Means Clustering

In [ ]:
def kmeans_with_speed_ratio(gdf, n_clusters=3):
    """
    Apply KMeans clustering using jamFactor, highway type, speed ratio, and spatial features.

    Parameters:
        gdf (GeoDataFrame): input GeoDataFrame with HERE API fields
        n_clusters (int): number of KMeans clusters

    Returns:
        GeoDataFrame: input with 'kmeans_cluster' column added
    """
    df = gdf.dropna(subset=["jamFactor", "highway", "currentSpeed", "freeFlowSpeed"]).copy()

    # Handle division and limit ratio
    df["speed_ratio"] = df["currentSpeed"] / df["freeFlowSpeed"]
    df["speed_ratio"] = df["speed_ratio"].clip(upper=1.0)  # cap at 1.0

    # Encode highway types
    highway_types = df["highway"].astype(str).unique()
    highway_map = {h: i for i, h in enumerate(sorted(highway_types))}
    df["highway_encoded"] = df["highway"].astype(str).map(highway_map)

    # Project to Web Mercator and extract coordinates
    df_proj = df.to_crs(epsg=3857)
    df_proj["x"] = df_proj.geometry.centroid.x
    df_proj["y"] = df_proj.geometry.centroid.y

    # Weight features
    df_proj["jamFactor_weighted"] = df_proj["jamFactor"] * 5.0
    df_proj["speed_ratio_weighted"] = df_proj["speed_ratio"] * 10.0
    df_proj["highway_encoded_weighted"] = df_proj["highway_encoded"] * 1.0

    # Assemble feature matrix
    X = df_proj[["x", "y", "jamFactor_weighted", "speed_ratio_weighted", "highway_encoded_weighted"]].values
    X_scaled = StandardScaler().fit_transform(X)

    # Apply KMeans
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
    df["kmeans_cluster"] = kmeans.fit_predict(X_scaled)

    return df

kmeans_speed_ratio_gdf = kmeans_with_speed_ratio(enriched_urban_traffic_gdf)

# Check how many per cluster
kmeans_speed_ratio_gdf["kmeans_cluster"].value_counts()


#Analyze K-Means clusters

In [ ]:
# Group by cluster to inspect average jam factor and common highway types
cluster_summary = (
    kmeans_speed_ratio_gdf
    .groupby("kmeans_cluster")
    .agg(
        segment_count=("jamFactor", "count"),
        avg_jamFactor=("jamFactor", "mean"),
        avg_speed_ratio=("speed_ratio", "mean"),
        most_common_highway=("highway", lambda x: x.mode()[0] if not x.mode().empty else "Unknown")
    )
    .reset_index()
)

# Dynamic congestion labeling function
def label_kmeans_congestion_by_cluster(df, cluster_col="kmeans_cluster"):
    df = df.copy()

    # Summarize congestion behavior per cluster
    summary = (
        df.groupby(cluster_col)
        .agg(
            avg_jamFactor=("jamFactor", "mean"),
            avg_speed_ratio=("speed_ratio", "mean")
        )
        .reset_index()
    )

    # Compute congestion score: higher jam = worse, lower speed = worse
    summary["congestion_score"] = (1 - summary["avg_speed_ratio"]) + summary["avg_jamFactor"]

    # Assign semantic labels based on score
    labels = ['Free Flowing', 'Moderate Flow', 'Congested']
    sorted_clusters = summary.sort_values("congestion_score")[cluster_col].values
    label_map = {cluster: labels[i] if i < len(labels) else f"Cluster {cluster}"
             for i, cluster in enumerate(sorted_clusters)}
    cluster_summary["congestion_label"] = cluster_summary["kmeans_cluster"].map(label_map)
    df["congestionLabel"] = df[cluster_col].map(label_map)
    return df

# Apply dynamic labeling to GeoDataFrame
labeled_kmeans_enriched_gdf = label_kmeans_congestion_by_cluster(kmeans_speed_ratio_gdf)

# Display updated summary
cluster_summary


# Generate Quartiles

In [ ]:
plt.figure(figsize=(10, 6))
labeled_kmeans_enriched_gdf["speed_ratio"].hist(bins=50, edgecolor="black")
plt.title("Distribution of Speed Ratio")
plt.xlabel("Speed Ratio (Current / Free Flow)")
plt.ylabel("Segment Count")
plt.grid(True)
plt.show()

# Custom bins
bins = [0, 0.4, 0.7, 0.9, 1.0]
labels = ["Heavy", "Moderate", "Light", "Free Flowing"]

# Apply binning
labeled_kmeans_enriched_gdf["congestion_quartile"] = pd.cut(
    labeled_kmeans_enriched_gdf["speed_ratio"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=True
)

# Add a numeric version
labeled_kmeans_enriched_gdf["congestion_quartile_num"] = pd.cut(
    labeled_kmeans_enriched_gdf["speed_ratio"],
    bins=bins,
    labels=False,
    include_lowest=True,
    right=True
)

# Simplify Data and Export to Drive

In [ ]:


# 1. Set local output path
output_filename = "test_congestion_3day_sample.geojson"
output_path = os.path.join(os.getcwd(), output_filename)

"""

# 2. Simplify and sample from full dataset
simplified_gdf = labeled_kmeans_enriched_gdf.copy()
simplified_gdf["geometry"] = simplified_gdf["geometry"].simplify(tolerance=0.0005, preserve_topology=True)

# Random sample of 50 road segments
sample_gdf = simplified_gdf.sample(n=50, random_state=2025).copy()

# 3. Create ISO 8601 UTC time strings for 8 AM PT on three days
def get_utc_time_string(year, month, day, hour, minute = 0):
    pacific = pytz.timezone("US/Pacific")
    dt_local = pacific.localize(datetime(year, month, day, hour, minute))
    dt_utc = dt_local.astimezone(pytz.utc)
    return dt_utc.strftime("%Y-%m-%dT%H:%M:%SZ")

# Adjusted timestamps: 8 AM and 5 PM PT for each date
timestamps = [
    get_utc_time_string(2025, 7, 19, 8),   # 2025-07-19T15:00:00Z
    get_utc_time_string(2025, 7, 19, 17),  # 2025-07-19T00:00:00Z (next day at midnight UTC)
    get_utc_time_string(2025, 7, 20, 8),
    get_utc_time_string(2025, 7, 20, 17),
    get_utc_time_string(2025, 7, 21, 8)
]


# 4. Assign one of three dates to each third of the rows
sample_gdf = sample_gdf.reset_index(drop=True)
sample_gdf["time"] = pd.Series([timestamps[i % len(timestamps)] for i in range(50)])

# 5. Define export columns
keep_fields = [
    "jamFactor", "speed_ratio", "highway", "roadName",
    "congestionLabel", "congestion_quartile", "congestion_quartile_num",
    "time", "geometry"
]
final_gdf = sample_gdf[keep_fields]

# 6. Save locally as GeoJSON
final_gdf.to_file(output_path, driver="GeoJSON")
print(f"Exported to: {output_path}")

# Get current time in Eastern Time
eastern = pytz.timezone("US/Eastern")
now_et = datetime.now(eastern)

# 1) Timestamps
snapshot_time_str = now_et.strftime("%Y-%m-%d %H:%M:%S")
snapshot_time_iso = now_et.isoformat()
timestamp_str = now_et.strftime("%Y-%m-%d_%H-%M")

df = labeled_kmeans_enriched_gdf.copy()

# Add timestamps
df["snapshot_time"] = snapshot_time_str
df["snapshot_time_iso"] = snapshot_time_iso

"""
eastern = pytz.timezone("US/Eastern")

def make_et_timestamp(hour: int, minute: int = 0):
    """
    Create a timezone-aware Eastern Time datetime using today's Eastern date
    and the specified hour/minute. This avoids UTC/date rollover issues.
    """
    today_et = datetime.now(eastern).date()  # today's date in ET
    # Build a naive dt with today's ET date, then localize to ET
    desired_naive = datetime(today_et.year, today_et.month, today_et.day, hour, minute, 0, 0)
    desired_et = eastern.localize(desired_naive)
    # Format
    snapshot_time_str = desired_et.strftime("%Y-%m-%d %H:%M:%S")
    snapshot_time_iso = desired_et.isoformat()
    timestamp_str = desired_et.strftime("%Y-%m-%d %H:%M:%S")
    return desired_et, snapshot_time_str, snapshot_time_iso, timestamp_str

# ---- pick  ET hour here ----
# 10 -> 10:00 ET  (≈ 7:00 PT during DST)
# 11 -> 11:00 ET  (≈ 8:00 PT during DST)
# 16 -> 7:00 PM ET (4:00 PM PT)
# 17 -> 8:00 PM ET (5:00 PM PT)
# 18 -> 9:00 PM ET (6:00 PM PT)
_, snapshot_time_str, snapshot_time_iso, timestamp_str = make_et_timestamp(hour=18)

# Apply to dataframe
df = labeled_kmeans_enriched_gdf.copy()
df["snapshot_time"] = timestamp_str
df["snapshot_time_iso"] = snapshot_time_iso

# 2) Numeric + compute composite score
df["speed_ratio"] = pd.to_numeric(df["speed_ratio"], errors="coerce").clip(upper=1.0)
df["jamFactor"]   = pd.to_numeric(df["jamFactor"],   errors="coerce")
df["congestion_score"] = (1 - df["speed_ratio"]) + df["jamFactor"]
df = df.dropna(subset=["congestion_score"])

# 3) Normalize 'highway'
def stringify_highway(val):
    if isinstance(val, (list, tuple)):
        return ", ".join(map(str, val)) if val else "Unknown"
    return "Unknown" if pd.isna(val) else str(val)

df["highway"] = df["highway"].apply(stringify_highway)

# 4) Simplify geometry
df = df.copy()
df["geometry"] = df["geometry"].simplify(tolerance=0.0005, preserve_topology=True)

# 5) Select fields
keep_fields = [
    "jamFactor", "speed_ratio", "highway", "roadName",
    "congestionLabel", "congestion_quartile", "congestion_quartile_num",
    "snapshot_time", "snapshot_time_iso",
    "congestion_score",
    "region",
    "geometry"
]
export_gdf = df[keep_fields]

# 6) Paths + export
historical_path = f"congestion_{timestamp_str}.geojson"
latest_path = "congestion_latest.geojson"

export_gdf.to_file(historical_path, driver="GeoJSON")
export_gdf.to_file(latest_path, driver="GeoJSON")

print(f"Files saved:\n- {historical_path}\n- {latest_path}")

# 6. Report file size
latest_mb = os.path.getsize(latest_path) / (1024 ** 2)
print(f"Exported {latest_path} ({latest_mb:.2f} MB)")


#

# Load and Prepare for Statistics

In [ ]:
# Load dataset
gdf = gpd.read_file("congestion_latest.geojson")

# Inspect columns
print(gdf.columns)

# Project to meters
gdf = gdf.to_crs(epsg=3857)

# Remove bad geometries  to connect graph
gdf = gdf[gdf.geometry.notnull()].copy()
gdf["cx"] = gdf.geometry.centroid.x
gdf["cy"] = gdf.geometry.centroid.y
gdf = gdf.drop_duplicates(subset=["cx","cy"]).copy()


# Generate and Compute Statistics by Region

In [ ]:
regions = sorted(gdf["region"].unique()) if "region" in gdf.columns else ["ALL"]
if regions == ["ALL"]:
    gdf["region"] = "ALL"

all_out = []

for r in regions:
    sub = gdf[gdf["region"] == r].copy()
    if len(sub) < 10:
        # too small for stable inference
        continue

    # Use centroids (lines -> points) for KNN
    sub["__cx__"] = sub.geometry.centroid.x
    sub["__cy__"] = sub.geometry.centroid.y

    # 4) Build KNN weights (adaptive: each feature gets k neighbors)
    #    k=8 is a common default; bump to 12 if network is sparse
    #    ids=sub.index preserves row alignment
    w = KNN.from_array(sub[["__cx__", "__cy__"]].to_numpy(), k=8, ids=sub.index)

    # Optional: check for islands
    # islands = [i for i, nbrs in w.neighbors.items() if len(nbrs) == 0]

    # 5) Global Moran’s I
    y = sub["congestion_score"].to_numpy()
    mi = Moran(y, w, permutations=9999)
    print(mi.I, mi.p_sim)


    # 6) Local Moran’s I (LISA)
    lisa = Moran_Local(y, w)
    sub["lisa_I"] = lisa.Is
    sub["lisa_p"] = lisa.p_sim
    sub["lisa_q"] = lisa.q

    def label(q, p):
        if p > 0.05:
            return "Not Significant"
        return {1:"High-High", 2:"Low-Low", 3:"High-Low", 4:"Low-High"}.get(q, "Not Significant")

    sub["lisa_cluster"] = [label(q, p) for q, p in zip(sub["lisa_q"], sub["lisa_p"])]

    print(f"[{r}] Global Moran's I = {mi.I:.3f}, p = {mi.p_sim:.4f}, n={len(sub)}")
    all_out.append(sub.drop(columns=["__cx__", "__cy__"]))

# 7) Concatenate and export with LISA results
if all_out:
    out = pd.concat(all_out).sort_index()
    out = out.to_crs(epsg=4326)  # back to WGS84 for web use
    out.to_file(f"congestion_lisa_{timestamp_str}.geojson", driver="GeoJSON")
    print("Wrote LISA results to congestion_lisa_knn.geojson")


# Visualize LISA outputs in Folium

In [ ]:

# 1) Load LISA results
path = f"congestion_lisa_{timestamp_str}.geojson"
assert os.path.exists(path), f"File not found: {path}. Run the LISA export first."

gdf = gpd.read_file(path)

# Safety: ensure we have expected columns
required_cols = {"lisa_cluster", "region", "congestion_score", "jamFactor", "speed_ratio"}
missing = required_cols - set(gdf.columns)
if missing:
    print(f"Warning: missing columns: {missing}")

# 2) Map center & base map
# If you have multiple regions, center over CA; otherwise fit to data below.
m = folium.Map(location=[36.8, -119.5], zoom_start=6, tiles="cartodbpositron")

# 3) Style: cluster -> color
colors = {
    "High-High": "red",
    "Low-Low": "blue",
    "High-Low": "orange",
    "Low-High": "green",
    "Not Significant": "lightgray",
}
def color_for(cluster):
    return colors.get(cluster, "lightgray")

# 4) Optional: one layer per region (nice for toggling in CARTO-like way)
use_region_layers = True and "region" in gdf.columns
region_values = sorted(gdf["region"].dropna().unique()) if use_region_layers else ["ALL"]
region_groups = {r: FeatureGroup(name=f"Region: {r}", show=True) for r in region_values}

# 5) Draw polylines (cap to avoid slow rendering in very large sets)
#    Increase 'max_draw' if your notebook can handle more.
max_draw = 20000
drawn = 0

# Fit bounds tracker
minx=miny=1e15; maxx=maxy=-1e15

for idx, row in gdf.iterrows():
    if drawn >= max_draw:
        break
    geom = row.geometry
    if geom is None or not isinstance(geom, LineString):
        continue

    cluster = str(row.get("lisa_cluster", "Not Significant"))
    clr = color_for(cluster)

    # Popup info (add or remove fields as desired)
    popup_html = folium.Html(f"""
    <div style="font-size:13px;">
      <b>Region:</b> {row.get('region','N/A')}<br>
      <b>Cluster:</b> {cluster}<br>
      <b>Congestion score:</b> {row.get('congestion_score','N/A')}<br>
      <b>Jam Factor:</b> {row.get('jamFactor','N/A')}<br>
      <b>Speed Ratio:</b> {row.get('speed_ratio','N/A')}<br>
      <b>Road:</b> {row.get('roadName','N/A')}<br>
      <b>Highway:</b> {row.get('highway','N/A')}
    </div>
    """, script=True)

    pl = folium.PolyLine(
        locations=[(lat, lon) for lon, lat in geom.coords],  # note lon/lat -> lat/lon
        color=clr, weight=3, opacity=0.8,
        popup=folium.Popup(popup_html, max_width=300)
    )

    # Add to appropriate group
    if use_region_layers:
        region_groups[row["region"]].add_child(pl)
    else:
        m.add_child(pl)

    # Update bounds
    bx, by, ex, ey = geom.bounds
    minx, miny = min(minx, bx), min(miny, by)
    maxx, maxy = max(maxx, ex), max(maxy, ey)

    drawn += 1

# 6) Add groups to map
if use_region_layers:
    for grp in region_groups.values():
        grp.add_to(m)

# 7) Add layer control
folium.LayerControl(collapsed=False).add_to(m)

# 8) Add a legend
legend_html = """
<div style="
 position: fixed;
 bottom: 20px; left: 20px; z-index: 9999;
 background: white; padding: 10px 12px; border: 1px solid #999; border-radius: 6px;
 box-shadow: 0 1px 4px rgba(0,0,0,0.3); font-size: 13px;">
  <b>LISA Cluster</b><br>
  <span style="color:red;">&#9632;</span> High-High<br>
  <span style="color:blue;">&#9632;</span> Low-Low<br>
  <span style="color:orange;">&#9632;</span> High-Low<br>
  <span style="color:green;">&#9632;</span> Low-High<br>
  <span style="color:gray;">&#9632;</span> Not Significant
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# 9) Fit map to data bounds if present
if drawn > 0:
    # bounds given as [[south, west], [north, east]]
    m.fit_bounds([[miny, minx], [maxy, maxx]])

# 10) Save & display
out_html = "lisa_clusters_map.html"
m.save(out_html)
m
print(f"Saved: {out_html}")


In [ ]:
# ---------------- CONFIG ----------------
path = "congestion_latest.geojson"
VALUE_FIELD_CANDIDATES = ["congestion_score", "composite_score"]
TARGET_REGION = None  # e.g., "Los Angeles" to filter; or None to plot all regions

# ---------------- LOAD ----------------
gdf = gpd.read_file(path)

# Pick the target value field
value_field = next((c for c in VALUE_FIELD_CANDIDATES if c in gdf.columns), None)
assert value_field is not None, f"None of {VALUE_FIELD_CANDIDATES} present. Columns: {list(gdf.columns)}"

# Ensure a region field exists even if not provided
if "region" not in gdf.columns:
    gdf["region"] = "ALL"

# Create a date field for summary output if one exists; otherwise use a placeholder
date_field = next((c for c in ["date", "snapshot_time", "capture_date", "datetime", "timestamp"] if c in gdf.columns), None)
if date_field is not None:
    gdf["summary_date"] = pd.to_datetime(gdf[date_field], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
    gdf["summary_date"] = gdf["summary_date"].fillna("Unknown")
else:
    gdf["summary_date"] = "Unknown"

# Optional: filter to a single region (e.g., Los Angeles)
if TARGET_REGION is not None:
    gdf = gdf[gdf["region"].str.contains(TARGET_REGION, case=False, na=False)].copy()
    assert len(gdf) > 0, f"No rows found for region filter: {TARGET_REGION}"

# Project to a metric CRS (Web Mercator is fine at city scale; a local CRS is even better)
gdf = gdf.to_crs(epsg=3857)

# --- Geometry hygiene to avoid disconnected graphs ---
gdf = gdf[gdf.geometry.notnull()].copy()
gdf = gdf[gdf.is_valid].copy()

# Remove exact duplicate centroids (common with linear features)
cent = gdf.geometry.centroid
gdf["cx"], gdf["cy"] = cent.x, cent.y
gdf = gdf.drop_duplicates(subset=["cx", "cy"]).copy()

# Output setup
summary_rows = []
plot_dir = "morans_scatterplots"
os.makedirs(plot_dir, exist_ok=True)

# Regions to iterate
regions = sorted(gdf["region"].unique())

for r in regions:
    sub = gdf[gdf["region"] == r].copy()
    if len(sub) < 10:
        # too small for stable inference
        continue

    # Build kNN weights on centroids (robust for lines/polygons)
    coords = sub[["cx", "cy"]].to_numpy()
    w = KNN.from_array(coords, k=8, ids=sub.index)
    w.transform = "R"  # row-standardize

    # Response vector
    y = sub[value_field].astype(float).to_numpy()

    # Global Moran
    mi = Moran(y, w, permutations=9999)

    # Save summary row
    summary_rows.append({
        "Region": r,
        "median Moran's I": mi.I,
        "Mean Moran's I": mi.I,
        "p-value": mi.p_sim,
        "n (segments)": len(sub),
        "Date": sub["summary_date"].mode().iloc[0] if len(sub["summary_date"].mode()) > 0 else "Unknown"
    })

    # ---- Moran scatterplot ----
    fig, ax = moran_scatterplot(mi, aspect_equal=True)
    ax.set_title(f"{r} — Moran's I Scatter\nI = {mi.I:.3f}, p = {mi.p_sim:.4f}", fontsize=12)
    ax.set_xlabel(value_field)
    ax.set_ylabel(f"Spatial lag of {value_field}")
    plt.tight_layout()

    safe_region = "".join(c if c.isalnum() or c in ("_", "-") else "_" for c in str(r))
    safe_date = "".join(c if c.isalnum() or c in ("_", "-") else "_" for c in str(summary_rows[-1]["Date"]))
    plot_path = os.path.join(plot_dir, f"moran_scatter_{safe_region}_{safe_date}.png")
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    # ---- Local Moran (optional, keeps your original logic) ----
    lisa = Moran_Local(y, w)
    sub["lisa_I"] = lisa.Is
    sub["lisa_p"] = lisa.p_sim
    sub["lisa_q"] = lisa.q
    # simple labeler
    sub["lisa_cluster"] = np.where(sub["lisa_p"] > 0.05, "Not Significant",
                                   np.where(sub["lisa_q"] == 1, "High-High",
                                   np.where(sub["lisa_q"] == 2, "Low-Low",
                                   np.where(sub["lisa_q"] == 3, "High-Low", "Low-High"))))

# Write summary CSV
summary_df = pd.DataFrame(summary_rows, columns=[
    "Region", "median Moran's I", "Mean Moran's I", "p-value", "n (segments)", "Date"
])
summary_csv = f"morans_i_{timestamp_str}_summary.csv"
summary_df.to_csv(summary_csv, index=False)

# Zip scatter plots
zip_path = "morans_scatterplots.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(plot_dir):
        fpath = os.path.join(plot_dir, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, arcname=fname)

